In [0]:


# COMMAND ----------
# Cria o schema da camada Silver
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# COMMAND ----------
# Importações usadas nas transformações
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# COMMAND ----------
# Dimensão de consumidores com deduplicação pelo registro mais recente
w_consumidor = Window.partitionBy("customer_id").orderBy(col("timestamp_ingestion").desc())

df_consumidores = (
    spark.table("bronze.tb_customers")
    .withColumn("rn", row_number().over(w_consumidor))
    .filter(col("rn") == 1)
    .select(
        col("customer_id").alias("id_consumidor"),
        col("customer_zip_code_prefix").alias("prefixo_cep"),
        col("customer_name").alias("nome_consumidor"),
        upper(col("customer_city")).alias("cidade"),
        upper(col("customer_state")).alias("estado")
    )
)

(
    df_consumidores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_consumidores")
)

# COMMAND ----------
# Fato de pedidos com tradução do status e métricas de entrega
df_pedidos = (
    spark.table("bronze.tb_orders")
    .withColumn(
        "status",
        when(col("order_status") == "delivered", "entregue")
        .when(col("order_status") == "canceled", "cancelado")
        .when(col("order_status") == "shipped", "enviado")
        .when(col("order_status") == "processing", "processando")
        .when(col("order_status") == "invoiced", "faturado")
        .when(col("order_status") == "unavailable", "indisponível")
        .when(col("order_status") == "created", "criado")
        .when(col("order_status") == "approved", "aprovado")
        .otherwise("desconhecido")
    )
    .withColumn("data_pedido", to_timestamp("order_purchase_timestamp"))
    .withColumn("data_entrega", to_timestamp("order_delivered_customer_date"))
    .withColumn("data_estimada_entrega", to_timestamp("order_estimated_delivery_date"))
    .withColumn("tempo_entrega_dias", datediff(col("data_entrega"), col("data_pedido")))
    .withColumn("tempo_entrega_estimado_dias", datediff(col("data_estimada_entrega"), col("data_pedido")))
    .withColumn("diferenca_entrega_dias", col("tempo_entrega_dias") - col("tempo_entrega_estimado_dias"))
    .withColumn(
        "entrega_no_prazo",
        when(col("status") != "entregue", "Não Entregue")
        .when(col("data_entrega") <= col("data_estimada_entrega"), "Sim")
        .otherwise("Não")
    )
    .select(
        col("order_id").alias("id_pedido"),
        col("customer_id").alias("id_consumidor"),
        col("status"),
        col("data_pedido"),
        col("data_entrega"),
        col("data_estimada_entrega"),
        col("tempo_entrega_dias"),
        col("tempo_entrega_estimado_dias"),
        col("diferenca_entrega_dias"),
        col("entrega_no_prazo")
    )
)

(
    df_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pedidos")
)

# COMMAND ----------
# Fato de itens de pedido com padronização dos nomes
df_itens = (
    spark.table("bronze.tb_order_items")
    .select(
        col("order_id").alias("id_pedido"),
        col("order_item_id").alias("id_item"),
        col("product_id").alias("id_produto"),
        col("seller_id").alias("id_vendedor"),
        round(col("price"), 2).alias("preco_brl"),
        round(col("freight_value"), 2).alias("preco_frete")
    )
)

(
    df_itens.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_itens_pedidos")
)

# COMMAND ----------
# Fato de pagamentos com tradução do tipo de pagamento
df_pagamentos = (
    spark.table("bronze.tb_order_payments")
    .withColumn(
        "tipo_pagamento",
        when(col("payment_type") == "credit_card", "Cartão de Crédito")
        .when(col("payment_type") == "boleto", "Boleto")
        .when(col("payment_type") == "voucher", "Voucher")
        .when(col("payment_type") == "debit_card", "Cartão de Débito")
        .when(col("payment_type") == "not_defined", "Não Definido")
        .otherwise("Não Definido")
    )
    .select(
        col("order_id").alias("id_pedido"),
        col("payment_sequential").alias("sequencial_pagamento"),
        col("tipo_pagamento"),
        col("payment_installments").alias("quantidade_parcelas"),
        round(col("payment_value"), 2).alias("valor_pagamento_brl")
    )
)

(
    df_pagamentos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pagamentos_pedidos")
)

# COMMAND ----------
# Avaliações: remove pedidos inválidos, trata datas e completa campos nulos
pedidos_validos = spark.table("silver.fat_pedidos").select("id_pedido").distinct()

df_avaliacoes = (
    spark.table("bronze.tb_order_reviews")
    .withColumn("data_criacao_avaliacao", expr("try_to_timestamp(review_creation_date)"))
    .withColumn("data_resposta_avaliacao", expr("try_to_timestamp(review_answer_timestamp)"))
    .withColumn(
        "titulo_avaliacao",
        when(trim(col("review_comment_title")).isNull() | (trim(col("review_comment_title")) == ""), "Sem título")
        .otherwise(col("review_comment_title"))
    )
    .withColumn(
        "comentario_avaliacao",
        when(trim(col("review_comment_message")).isNull() | (trim(col("review_comment_message")) == ""), "Sem comentário")
        .otherwise(col("review_comment_message"))
    )
    .join(pedidos_validos, col("order_id") == col("id_pedido"), "inner")
    .filter(col("data_criacao_avaliacao").isNull() | (to_date(col("data_criacao_avaliacao")) <= current_date()))
    .select(
        col("review_id").alias("id_avaliacao"),
        col("order_id").alias("id_pedido"),
        col("review_score").cast("int").alias("nota_avaliacao"),
        col("titulo_avaliacao"),
        col("comentario_avaliacao"),
        col("data_criacao_avaliacao"),
        col("data_resposta_avaliacao")
    )
)

(
    df_avaliacoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_avaliacoes_pedidos")
)

# COMMAND ----------
# Dimensão de produtos com deduplicação pelo registro mais recente
w_produtos = Window.partitionBy("product_id").orderBy(col("timestamp_ingestion").desc())

df_produtos = (
    spark.table("bronze.tb_products")
    .withColumn("rn", row_number().over(w_produtos))
    .filter(col("rn") == 1)
    .select(
        col("product_id").alias("id_produto"),
        col("product_name").alias("nome_produto"),
        col("product_category_name").alias("categoria_produto"),
        col("product_weight_g").alias("peso_produto_gramas"),
        col("product_length_cm").alias("comprimento_centimetros"),
        col("product_height_cm").alias("altura_centimetros"),
        col("product_width_cm").alias("largura_centimetros"),
        col("product_photos_qty").alias("quantidade_fotos"),
        col("product_name_lenght").alias("tamanho_nome_produto"),
        col("product_description_lenght").alias("tamanho_descricao_produto")
    )
)

(
    df_produtos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_produtos")
)

# COMMAND ----------
# Dimensão de vendedores com deduplicação e padronização de cidade/estado
w_vendedores = Window.partitionBy("seller_id").orderBy(col("timestamp_ingestion").desc())

df_vendedores = (
    spark.table("bronze.tb_sellers")
    .withColumn("rn", row_number().over(w_vendedores))
    .filter(col("rn") == 1)
    .select(
        col("seller_id").alias("id_vendedor"),
        col("seller_name").alias("nome_vendedor"),
        col("seller_zip_code_prefix").alias("prefixo_cep"),
        upper(col("seller_city")).alias("cidade"),
        upper(col("seller_state")).alias("estado")
    )
)

(
    df_vendedores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_vendedores")
)

# COMMAND ----------
# Dimensão de tradução das categorias de produto
df_categoria = (
    spark.table("bronze.tb_product_category_name_translation")
    .select(
        col("product_category_name").alias("nome_produto_pt"),
        col("product_category_name_english").alias("nome_produto_en")
    )
)

(
    df_categoria.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_categoria_produtos_traducao")
)

# COMMAND ----------
# Cria calendário contínuo de cotação e preenche finais de semana com a última cotação válida
df_dolar_raw = (
    spark.table("bronze.tb_cotacao_dolar")
    .withColumn("data_cotacao_ts", to_timestamp("dataHoraCotacao"))
    .withColumn("data_cotacao", to_date("data_cotacao_ts"))
    .groupBy("data_cotacao")
    .agg(max("cotacaoCompra").alias("cotacao_dolar"))
)

datas = df_dolar_raw.agg(
    min("data_cotacao").alias("min_data"),
    max("data_cotacao").alias("max_data")
).collect()[0]

min_data = datas["min_data"]
max_data = datas["max_data"]

calendario = spark.sql(f"""
SELECT explode(sequence(to_date('{min_data}'), to_date('{max_data}'), interval 1 day)) AS data_cotacao
""")

df_calendario_dolar = calendario.join(df_dolar_raw, "data_cotacao", "left")

w_dolar = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)

df_dolar = (
    df_calendario_dolar
    .withColumn("cotacao_dolar_preenchida", last("cotacao_dolar", ignorenulls=True).over(w_dolar))
    .select(
        col("data_cotacao"),
        round(col("cotacao_dolar_preenchida"), 4).alias("cotacao_dolar")
    )
)

(
    df_dolar.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_cotacao_dolar")
)

# COMMAND ----------
# Consolida o valor total pago por pedido e converte para USD
df_pag_total = (
    spark.table("silver.fat_pagamentos_pedidos")
    .groupBy("id_pedido")
    .agg(round(sum("valor_pagamento_brl"), 2).alias("valor_total_pago_brl"))
)

df_pedido_total = (
    spark.table("silver.fat_pedidos").alias("p")
    .join(df_pag_total.alias("pg"), "id_pedido", "left")
    .join(
        spark.table("silver.dim_cotacao_dolar").alias("d"),
        to_date(col("p.data_pedido")) == col("d.data_cotacao"),
        "left"
    )
    .select(
        col("p.id_pedido"),
        col("p.id_consumidor"),
        col("p.status"),
        round(col("pg.valor_total_pago_brl"), 2).alias("valor_total_pago_brl"),
        round(col("pg.valor_total_pago_brl") / col("d.cotacao_dolar"), 2).alias("valor_total_pago_usd"),
        col("p.data_pedido")
    )
)

(
    df_pedido_total.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fat_pedido_total")
)

# COMMAND ----------
# Otimiza as tabelas fato mais importantes para consulta analítica
spark.sql("OPTIMIZE silver.fat_pedidos ZORDER BY (id_pedido, data_pedido)")
spark.sql("OPTIMIZE silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")

# COMMAND ----------
# Lista as tabelas criadas na camada Silver
spark.sql("SHOW TABLES IN silver").display()